# Data and baseline

In [1]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_DIR = Path.cwd() / 'scratchpad/hyperliquid-ic' if (Path.cwd() / 'scratchpad/hyperliquid-ic').exists() else Path.cwd()
sys.path.insert(0, str(PROJECT_DIR))
ARTIFACTS = PROJECT_DIR / '_artifacts-rewrite'
ARTIFACTS.mkdir(exist_ok=True)
from ic_research import ResearchConfig, completed_daily_window, config_dict, fetch_btc_reference, load_source_data, prepare_observations, write_json

CONFIG = ResearchConfig()
raw, metadata, source_info = load_source_data(config=CONFIG)
observations = prepare_observations(raw)
btc_reference = fetch_btc_reference(CONFIG)
evaluation_start, evaluation_end = completed_daily_window(observations)

raw.to_parquet(ARTIFACTS / 'raw-observations.parquet', index=False)
observations.to_parquet(ARTIFACTS / 'observations.parquet', index=False)
btc_reference.to_parquet(ARTIFACTS / 'btc-reference.parquet')
metadata.to_csv(ARTIFACTS / 'vault-metadata.csv')
write_json(ARTIFACTS / 'config.json', {'config': config_dict(CONFIG), 'source': source_info, 'evaluation_start': str(evaluation_start), 'evaluation_end': str(evaluation_end), 'decision_timestamp_contract': 'features use fresh observations before date + 1 day 00:00 UTC; labels enter at first fresh mark at or after that timestamp'})
display(pd.DataFrame([source_info]))
display(pd.DataFrame({'evaluation_start': [evaluation_start], 'evaluation_end': [evaluation_end], 'calendar_decision_dates': [(evaluation_end - evaluation_start).days + 1]}))


,refresh,source_path,price_rows,vaults,observation_start,observation_end,written_at_start,written_at_end,historical_point_in_time_reconstructable,research_contract
0,False,/Users/moo/.tradingstrategy/vaults/downloads/v...,1495861,602,2023-03-01 00:00:00,2026-09-13 13:50:25.069000,2026-03-23 01:09:35.775000,2026-09-13 13:50:32.437000,False,current reconstructed provider history; writte...


,evaluation_start,evaluation_end,calendar_decision_dates
0,2025-09-13,2026-09-12,365


In [2]:
audit = observations.groupby('address').agg(first_observation=('timestamp', 'min'), last_observation=('timestamp', 'max'), fresh_marks=('timestamp', 'size'), latest_tvl=('total_assets', 'last'))
audit['available_history_days'] = (audit.last_observation - audit.first_observation).dt.total_seconds() / 86400
audit['age_cohort'] = pd.cut(audit.available_history_days, [-1, 7, 14, 30, 90, 180, 360, float('inf')], labels=['<7', '7-13', '14-29', '30-89', '90-179', '180-359', '360+'])
audit.to_parquet(ARTIFACTS / 'data-audit.parquet')
coverage = observations.assign(date=observations.timestamp.dt.normalize()).groupby('date').agg(fresh_marks=('address', 'size'), vaults=('address', 'nunique')).reset_index()
coverage.to_parquet(ARTIFACTS / 'fresh-observation-coverage.parquet', index=False)
display(audit.groupby('age_cohort', observed=False).agg(vaults=('fresh_marks', 'size'), median_marks=('fresh_marks', 'median'), median_history_days=('available_history_days', 'median')).reset_index())
display(coverage[coverage.date.between(evaluation_start, evaluation_end)].describe())
print('Provider-carried rows retained in raw audit but excluded from fresh observation features:', int((~raw.is_fresh).sum()))


,age_cohort,vaults,median_marks,median_history_days
0,<7,7,5.0,4.000000
1,7-13,3,12.0,11.000000
2,14-29,26,29.5,24.478580
3,30-89,51,67.0,58.000000
4,90-179,95,883.0,157.577732
5,180-359,212,1053.5,241.631042
6,360+,208,1140.5,508.565996


,date,fresh_marks,vaults
count,257,257.000000,257.000000
mean,2026-05-03 06:54:37.821000,1851.346304,414.412451
min,2025-09-17 00:00:00,2.000000,2.000000
25%,2026-03-04 00:00:00,613.000000,388.000000
50%,2026-05-07 00:00:00,2400.000000,449.000000
75%,2026-07-10 00:00:00,2777.000000,477.000000
max,2026-09-12 00:00:00,2976.000000,538.000000
std,NaN,1074.959629,87.588221


Provider-carried rows retained in raw audit but excluded from fresh observation features: 1008123
